In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

Setting up session and reading data from cloud(s3)

In [ ]:
from config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

customers_df = spark.read.csv(
    s3_path("bronze", "customers", "olist_customers_dataset.csv"),
    header=True,
    inferSchema=True
)

customers_df.show(5)

understand the data even before cleaning

In [ ]:
print(f"column names:{customers_df.columns}")
print(f"number of colums:{len(customers_df.columns)}")
print(f"Total reocrds: {customers_df.count()}")
customers_df.printSchema()


In [ ]:
customers_df.columns

cid = (customers_df.groupBy("customer_id").count().filter("count > 1"))
cid1 = (customers_df.groupBy("customer_unique_id").count().filter("count > 1"))


print(cid.count())
print(cid1.count())
customers_df.select("customer_city").distinct().count()

In [ ]:
from pyspark.sql.functions import col, count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

In [ ]:
print("Original :", customers_df.count())

print("Distinct :", customers_df.dropDuplicates().count())

In [ ]:
customers_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("C:/Automated-cloud-batch processing/Silver/customers")